# Task 2.2 — Reproduction of Contribution from the Selected Paper (20 marks)

**Paper:** Kernel Latent SVM for Visual Recognition (NeurIPS 2012)

---

## Contribution Being Reproduced

We reproduce the **object classification with latent subcategories** experiment from **Section 4.2** of the paper. The key contribution is that Kernel Latent SVM (KLSVM) outperforms both linear LSVM and non-latent kernel SVM by combining nonlinear kernels with latent subcategory discovery.

## Evaluation Metric

**Multi-class classification accuracy (%)**, consistent with Table 2 in the paper.

In [1]:
# ============================================================
# Setup: Load data from Task 2.1
# ============================================================
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from skimage.feature import hog
from skimage.color import rgb2gray
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Hyperparameters (same as Task 2.1)
NUM_CLASSES_SUBSET = 4
SAMPLES_PER_CLASS_TRAIN = 200
SAMPLES_PER_CLASS_TEST = 100
NUM_SUBCATEGORIES = 3  # |H_c| = 3 (Section 4.2)
C_PARAM = 0.01  # C1 = C2 = 0.01 (Section 4.2)
MAX_ITER = 5
HOG_ORIENTATIONS = 8
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
SELECTED_CLASSES = [0, 1, 2, 3]
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat']

# Load and preprocess
print('Loading CIFAR-10...')
# ============================================================
# Robust CIFAR-10 Loading
# ============================================================
import pickle
import urllib.request
import tarfile
import os
import shutil

def load_cifar10(data_dir='/tmp/cifar10'):
    """Download and load CIFAR-10 dataset directly from source."""
    url = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
    tar_path = os.path.join(data_dir, 'cifar-10-python.tar.gz')
    extract_dir = os.path.join(data_dir, 'cifar-10-batches-py')
    
    if not os.path.exists(extract_dir):
        os.makedirs(data_dir, exist_ok=True)
        print('Downloading CIFAR-10...')
        urllib.request.urlretrieve(url, tar_path)
        print('Extracting...')
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(data_dir)
        os.remove(tar_path)
    
    # Load training batches
    X_train_list, y_train_list = [], []
    for i in range(1, 6):
        batch_file = os.path.join(extract_dir, f'data_batch_{i}')
        with open(batch_file, 'rb') as f:
            batch = pickle.load(f, encoding='bytes')
        X_train_list.append(batch[b'data'])
        y_train_list.extend(batch[b'labels'])
    
    # Load test batch
    test_file = os.path.join(extract_dir, 'test_batch')
    with open(test_file, 'rb') as f:
        test_batch = pickle.load(f, encoding='bytes')
    
    X_all = np.vstack(X_train_list + [test_batch[b'data']]).astype(np.float32) / 255.0
    y_all = np.array(y_train_list + test_batch[b'labels'], dtype=int)
    
    print(f'CIFAR-10 loaded: {X_all.shape[0]} samples, {X_all.shape[1]} features')
    return X_all, y_all

print('Loading CIFAR-10...')
X_raw, y_raw = load_cifar10()
print(f'Total samples: {X_raw.shape[0]}')
print(f'Classes: {np.unique(y_raw)}')

def extract_hog_features(images_flat, img_shape=(32, 32, 3)):
    features = []
    for img_flat in images_flat:
        img = img_flat.reshape(img_shape)
        img_gray = rgb2gray(img)
        feat = hog(img_gray, orientations=HOG_ORIENTATIONS,
                   pixels_per_cell=HOG_PIXELS_PER_CELL,
                   cells_per_block=HOG_CELLS_PER_BLOCK,
                   feature_vector=True)
        features.append(feat)
    return np.array(features)

train_idx, test_idx = [], []
for cls in SELECTED_CLASSES:
    cls_indices = np.where(y_raw == cls)[0]
    np.random.shuffle(cls_indices)
    train_idx.extend(cls_indices[:SAMPLES_PER_CLASS_TRAIN])
    test_idx.extend(cls_indices[SAMPLES_PER_CLASS_TRAIN:SAMPLES_PER_CLASS_TRAIN + SAMPLES_PER_CLASS_TEST])

X_train_hog = normalize(extract_hog_features(X_raw[train_idx]), norm='l2')
y_train = y_raw[train_idx]
X_test_hog = normalize(extract_hog_features(X_raw[test_idx]), norm='l2')
y_test = y_raw[test_idx]

print(f'Training: {X_train_hog.shape}, Test: {X_test_hog.shape}')

Loading CIFAR-10...
Loading CIFAR-10...
CIFAR-10 loaded: 60000 samples, 3072 features
Total samples: 60000
Classes: [0 1 2 3 4 5 6 7 8 9]
Training: (800, 288), Test: (400, 288)


We reload and preprocess the data identically to Task 2.1. HOG features are extracted and L2-normalized as described in the paper (Sections 3.1 and 4.2).

In [2]:
# ============================================================
# Histogram Intersection Kernel (HIK)
# Paper Section 4.1-4.3: "We use the histogram intersection kernel"
# HIK(x, x') = sum_d min(x_d, x'_d)
# ============================================================

def histogram_intersection_kernel(X, Y):
    """Compute HIK kernel matrix between X and Y.
    
    Corresponds to the kernel choice in Sections 4.1, 4.2, 4.3.
    HIK is known to be a PSD kernel (Maji et al., 2008).
    """
    # Use broadcasting for efficiency
    K = np.zeros((X.shape[0], Y.shape[0]))
    for i in range(X.shape[0]):
        K[i] = np.minimum(X[i], Y).sum(axis=1)
    return K

print('HIK kernel function defined.')
# Test on small subset
K_test = histogram_intersection_kernel(X_train_hog[:5], X_train_hog[:5])
print(f'Test kernel matrix (5x5):\n{K_test}')

HIK kernel function defined.
Test kernel matrix (5x5):
[[14.53809738 11.02549648 11.09277344 11.68845654  9.32548141]
 [11.02549648 13.52260017 10.88177586 11.30310822  8.30107117]
 [11.09277344 10.88177586 14.58899784 11.59577942  9.81905174]
 [11.68845654 11.30310822 11.59577942 14.39370155  9.45887375]
 [ 9.32548141  8.30107117  9.81905174  9.45887375 14.9247961 ]]


The Histogram Intersection Kernel (HIK) is defined as $k(x, x') = \sum_d \min(x_d, x'_d)$. The paper uses HIK throughout all experiments (Sections 4.1–4.3), citing Maji et al. (2008) who showed HIK is PSD and achieves strong performance for visual recognition. This kernel satisfies the PSD requirement from Assumption 3 (Task 1.2).

In [3]:
# ============================================================
# Latent Subcategory Feature Construction
# Paper Section 4.2: phi(x, h) is a sparse vector
# phi(x, h=1) = (phi(x); 0; 0)
# phi(x, h=2) = (0; phi(x); 0)
# phi(x, h=3) = (0; 0; phi(x))
# ============================================================

def build_subcategory_features(X, h_labels, num_subcats):
    """Build sparse subcategory feature vectors phi(x, h).
    
    As described in Section 4.2: the feature dimension is |H_c| times
    the dimension of phi(x). phi(x, h=k) has phi(x) in the k-th block
    and zeros elsewhere.
    """
    n_samples, feat_dim = X.shape
    X_expanded = np.zeros((n_samples, feat_dim * num_subcats))
    for i in range(n_samples):
        h = int(h_labels[i])
        X_expanded[i, h * feat_dim:(h + 1) * feat_dim] = X[i]
    return X_expanded

print(f'Feature dim before expansion: {X_train_hog.shape[1]}')
print(f'Feature dim after expansion: {X_train_hog.shape[1] * NUM_SUBCATEGORIES}')

Feature dim before expansion: 288
Feature dim after expansion: 864


This implements the sparse subcategory feature representation from Section 4.2. For a sample assigned to subcategory $h \in \{0, 1, 2\}$, the feature vector $\phi(x, h)$ places the original HOG features in the $h$-th block and zeros elsewhere. This creates subcategory-specific templates, allowing the SVM to learn different decision boundaries for different visual "modes" within each class.

In [4]:
# ============================================================
# KLSVM: Kernel Latent SVM with Latent Subcategories
# Implements the alternating optimization from Section 3:
#   (a) Fix alpha, beta -> optimize {h_i} (Eq. 6)
#   (b) Fix {h_i} -> optimize alpha, beta via kernel SVM (Eq. 7)
# ============================================================

def klsvm_train(X_train, y_train, num_subcats, C, max_iter, kernel_func):
    """Train KLSVM with latent subcategories.
    
    Algorithm from Section 3, applied to the subcategory setting of Section 4.2.
    Uses one-vs-one multi-class strategy.
    """
    n_samples = X_train.shape[0]
    unique_classes = np.unique(y_train)
    
    # Step 1: Initialize subcategory labels using k-means clustering
    # Paper Section 4.2: "initialize the subcategory labels by k-means clustering"
    h_labels = np.zeros(n_samples, dtype=int)
    for cls in unique_classes:
        cls_mask = y_train == cls
        cls_features = X_train[cls_mask]
        kmeans = KMeans(n_clusters=num_subcats, random_state=RANDOM_SEED, n_init=10)
        h_labels[cls_mask] = kmeans.fit_predict(cls_features)
    
    print(f'  Initial subcategory distribution: {np.bincount(h_labels)}')
    
    best_acc = 0
    best_svm = None
    best_h = h_labels.copy()
    
    for iteration in range(max_iter):
        # Step (b): Fix {h_i}, train kernel SVM (Eq. 7)
        # Build expanded features phi(x, h)
        X_expanded = build_subcategory_features(X_train, h_labels, num_subcats)
        
        # Compute kernel matrix for expanded features
        K_train = kernel_func(X_expanded, X_expanded)
        
        # Train SVM with precomputed kernel
        svm = SVC(kernel='precomputed', C=C, decision_function_shape='ovo')
        svm.fit(K_train, y_train)
        
        # Evaluate on training set
        train_pred = svm.predict(K_train)
        train_acc = accuracy_score(y_train, train_pred)
        
        print(f'  Iter {iteration + 1}: train_acc = {train_acc:.4f}')
        
        if train_acc > best_acc:
            best_acc = train_acc
            best_svm = svm
            best_h = h_labels.copy()
        
        # Step (a): Fix SVM, update {h_i} for each positive example
        # For each training example, try all subcategory assignments
        # and pick the one that maximizes the decision function (Eq. 6/9)
        h_changed = 0
        for i in range(n_samples):
            best_score = -np.inf
            best_h_i = h_labels[i]
            
            for h_candidate in range(num_subcats):
                # Create temporary expanded feature with candidate h
                x_temp = np.zeros(X_train.shape[1] * num_subcats)
                x_temp[h_candidate * X_train.shape[1]:(h_candidate + 1) * X_train.shape[1]] = X_train[i]
                
                # Compute kernel with all training points
                k_vec = kernel_func(x_temp.reshape(1, -1), X_expanded)[0]
                
                # Compute decision function value
                score = svm.decision_function(k_vec.reshape(1, -1))
                score_val = np.sum(np.abs(score))  # Margin proxy
                
                if score_val > best_score:
                    best_score = score_val
                    best_h_i = h_candidate
            
            if best_h_i != h_labels[i]:
                h_changed += 1
            h_labels[i] = best_h_i
        
        print(f'  Iter {iteration + 1}: {h_changed} subcategory labels changed')
        
        if h_changed == 0:
            print(f'  Converged at iteration {iteration + 1}')
            break
    
    return best_svm, best_h, X_expanded

print('KLSVM training function defined.')

KLSVM training function defined.


This is the core KLSVM implementation. The algorithm follows Section 3 of the paper:

1. **Initialize** subcategory labels via k-means (Section 4.2: "initialize the subcategory labels of training images by k-means clustering")
2. **Iterate:**
   - **Step (b):** Fix subcategory assignments $\{h_i\}$, build expanded features $\phi(x, h)$, compute HIK kernel matrix, and train kernel SVM (corresponds to Eq. 7)
   - **Step (a):** Fix SVM parameters, update each $h_i$ by trying all subcategory assignments and keeping the one that maximizes the decision function margin (corresponds to Eq. 6/9)
3. **Converge** when no subcategory labels change

In [5]:
# ============================================================
# Train KLSVM
# ============================================================
print('Training KLSVM with HIK kernel and latent subcategories...')
print(f'Settings: C={C_PARAM}, K={NUM_SUBCATEGORIES} subcategories, max_iter={MAX_ITER}')

klsvm_model, final_h_labels, X_train_expanded = klsvm_train(
    X_train_hog, y_train, NUM_SUBCATEGORIES, C_PARAM, MAX_ITER,
    histogram_intersection_kernel
)

print(f'\nFinal subcategory distribution: {np.bincount(final_h_labels)}')

Training KLSVM with HIK kernel and latent subcategories...
Settings: C=0.01, K=3 subcategories, max_iter=5
  Initial subcategory distribution: [246 301 253]
  Iter 1: train_acc = 0.6062
  Iter 1: 493 subcategory labels changed
  Iter 2: train_acc = 0.4313
  Iter 2: 302 subcategory labels changed
  Iter 3: train_acc = 0.4375
  Iter 3: 486 subcategory labels changed
  Iter 4: train_acc = 0.4662
  Iter 4: 593 subcategory labels changed
  Iter 5: train_acc = 0.4625
  Iter 5: 661 subcategory labels changed

Final subcategory distribution: [246 301 253]


The KLSVM training runs the alternating optimization. We use `C1 = C2 = 0.01` and `|H_c| = 3` subcategories per class, matching the paper's settings in Section 4.2.

In [6]:
# ============================================================
# Evaluate KLSVM on Test Set
# ============================================================

def klsvm_predict(svm_model, X_train_expanded, X_test, h_labels_test_init,
                  num_subcats, kernel_func):
    """Predict using KLSVM.
    
    For test images, we try all subcategory assignments and pick
    the one with the highest decision function margin.
    Paper: f(x_new) = max_{h_new} score(x_new, h_new)
    """
    n_test = X_test.shape[0]
    feat_dim = X_test.shape[1]
    predictions = []
    
    for i in range(n_test):
        best_pred = None
        best_margin = -np.inf
        
        for h_candidate in range(num_subcats):
            x_temp = np.zeros(feat_dim * num_subcats)
            x_temp[h_candidate * feat_dim:(h_candidate + 1) * feat_dim] = X_test[i]
            
            k_vec = kernel_func(x_temp.reshape(1, -1), X_train_expanded)
            pred = svm_model.predict(k_vec)
            df = svm_model.decision_function(k_vec)
            margin = np.max(np.abs(df))
            
            if margin > best_margin:
                best_margin = margin
                best_pred = pred[0]
        
        predictions.append(best_pred)
    
    return np.array(predictions)

print('Evaluating KLSVM on test set...')
y_pred_klsvm = klsvm_predict(
    klsvm_model, X_train_expanded, X_test_hog,
    None, NUM_SUBCATEGORIES, histogram_intersection_kernel
)
klsvm_accuracy = accuracy_score(y_test, y_pred_klsvm)
print(f'KLSVM Test Accuracy: {klsvm_accuracy * 100:.2f}%')

Evaluating KLSVM on test set...
KLSVM Test Accuracy: 36.50%


For test-time prediction, we follow the kernelized scoring function from the end of Section 2: $f(x_{new}) = \max_{h_{new}} \text{score}(x_{new}, h_{new})$. For each test image, we try all $K$ subcategory assignments and select the one with the highest decision function margin, then report the predicted class for that assignment.

In [7]:
# ============================================================
# Baselines for Comparison
# ============================================================

# Baseline 1: BOF + Linear SVM (no latent variables)
print('Training Baseline 1: Linear SVM (no latent variables)...')
svm_linear = SVC(kernel='linear', C=C_PARAM, decision_function_shape='ovo')
svm_linear.fit(X_train_hog, y_train)
y_pred_linear = svm_linear.predict(X_test_hog)
linear_accuracy = accuracy_score(y_test, y_pred_linear)
print(f'Linear SVM Accuracy: {linear_accuracy * 100:.2f}%')

# Baseline 2: BOF + Kernel SVM (HIK, no latent variables)
print('\nTraining Baseline 2: Kernel SVM with HIK (no latent variables)...')
K_train_full = histogram_intersection_kernel(X_train_hog, X_train_hog)
K_test_full = histogram_intersection_kernel(X_test_hog, X_train_hog)
svm_kernel = SVC(kernel='precomputed', C=C_PARAM, decision_function_shape='ovo')
svm_kernel.fit(K_train_full, y_train)
y_pred_kernel = svm_kernel.predict(K_test_full)
kernel_accuracy = accuracy_score(y_test, y_pred_kernel)
print(f'Kernel SVM (HIK) Accuracy: {kernel_accuracy * 100:.2f}%')

# Baseline 3: Linear LSVM (linear kernel + latent subcategories)
print('\nTraining Baseline 3: Linear LSVM (linear + latent subcategories)...')
X_train_expanded_linear = build_subcategory_features(X_train_hog, final_h_labels, NUM_SUBCATEGORIES)
svm_lsvm = SVC(kernel='linear', C=C_PARAM, decision_function_shape='ovo')
svm_lsvm.fit(X_train_expanded_linear, y_train)
# For LSVM test, use same h inference but with linear kernel
best_preds = []
for i in range(len(X_test_hog)):
    best_pred_i = None
    best_margin_i = -np.inf
    for h in range(NUM_SUBCATEGORIES):
        x_temp = np.zeros(X_test_hog.shape[1] * NUM_SUBCATEGORIES)
        x_temp[h * X_test_hog.shape[1]:(h + 1) * X_test_hog.shape[1]] = X_test_hog[i]
        df = svm_lsvm.decision_function(x_temp.reshape(1, -1))
        margin = np.max(np.abs(df))
        if margin > best_margin_i:
            best_margin_i = margin
            best_pred_i = svm_lsvm.predict(x_temp.reshape(1, -1))[0]
    best_preds.append(best_pred_i)
lsvm_accuracy = accuracy_score(y_test, np.array(best_preds))
print(f'Linear LSVM Accuracy: {lsvm_accuracy * 100:.2f}%')

print('\n' + '='*50)
print('SUMMARY OF RESULTS')
print('='*50)
print(f'Linear SVM (no latent):    {linear_accuracy * 100:.2f}%')
print(f'Kernel SVM HIK (no latent):{kernel_accuracy * 100:.2f}%')
print(f'Linear LSVM (latent):      {lsvm_accuracy * 100:.2f}%')
print(f'KLSVM (ours):              {klsvm_accuracy * 100:.2f}%')

Training Baseline 1: Linear SVM (no latent variables)...
Linear SVM Accuracy: 49.25%

Training Baseline 2: Kernel SVM with HIK (no latent variables)...
Kernel SVM (HIK) Accuracy: 47.75%

Training Baseline 3: Linear LSVM (linear + latent subcategories)...
Linear LSVM Accuracy: 39.00%

SUMMARY OF RESULTS
Linear SVM (no latent):    49.25%
Kernel SVM HIK (no latent):47.75%
Linear LSVM (latent):      39.00%
KLSVM (ours):              36.50%


We compare against three baselines, matching the paper's experimental setup in Section 4.2:

1. **Linear SVM** — standard linear SVM on HOG features, no latent variables (equivalent to "BOF + linear SVM" in Table 2)
2. **Kernel SVM (HIK)** — kernel SVM using histogram intersection kernel, no latent variables (equivalent to "BOF + kernel SVM" in Table 2)
3. **Linear LSVM** — linear SVM with latent subcategory features (equivalent to "linear LSVM" in Table 2)

The paper's Table 2 reports: Linear SVM < Kernel SVM < Linear LSVM < KLSVM.